In [ ]:
from PIL import Image
import os
import json
from textwrap import dedent
import pandas as pd
#my spesific setting
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph
from pegiguard.data.prompts import build_prompt
from llavaguard_config import local_data_dir

In [1]:

def build_prompt_nudity3(category: str, subcategory: str, graph_info):
    edge = graph_info.get("edge")
    examples = graph_info.get("examples")
    rating = graph_info.get("rating")

    subcat_rules = {
        "15. Educational Content": dedent("""\
            Decide YES only if the image is clearly educational, historical, or informational about nudity.
            
        """),
        "16. Sexual Innuendo": dedent("""\
            Decide YES if the image conveys suggestiveness without explicit exposure.
        """),
        "17. Mild Nudity Content": dedent("""\
            In examples “exposed breasts” means at least one VISIBLE nipple or areola on a female-presenting chest.
            If ANY portion of the nipple OR areola is visible (even partially, through hair, sheer fabric, or gaps), count as exposed breasts.
            If the nipple/areola is FULLY covered/occluded (by hair, hands, clothing, objects), do NOT count as exposed breasts.

        """),
        "18. Explicit Sexual Content": dedent("""\
            In examples “visible genitalia” requires any part of the genitals to be visible to the viewer (even partially).
            Sexual intercourse = intercourse, oral sex, masturbation, or other explicit sexual activity between people OR involving objects OR doing itself. 
            Nudity alone without a sexual act is NOT “sexual content”.

        """),
        "19. Explicit Nudity Content": dedent("""\
        """),
        "20. Monetized Nudity": dedent("""\
            Decide YES when nudity or sexual content is presented as pornography or promotes sexual services.
            Humor, or suggestive references to sex without visible nudity/acts are NOT monetized nudity.  
        """)
    }

    op_rules = dedent("""\
        Use ONLY visible pixels. Ignore captions, filenames, off-screen context, and assumed intent. Do not use facial expression ALONE to decide.
        Default decision is "NO".
    """)

    rules = subcat_rules.get(subcategory, "Decide YES only if the visible content clearly matches the definition and examples for this subcategory.")

    prompt = dedent(f"""\
        Your task: Decide whether the image relates to the specified subcategory based ONLY on visible evidence.

        Subcategory to assess:
        - Top-level Category: {category}
        - Subcategory: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples (illustrative, not exhaustive): {examples}

        Operational rules:
        {op_rules}

        Subcategory-specific decision guide:
        {rules}

        Output requirement:
        Return EXACTLY "YES" if the image clearly matches this subcategory based on visible evidence; otherwise return EXACTLY "NO".
        If evidence is ambiguous or uncertain, then return "NO".
    """).strip()

    return prompt


In [2]:
# 2. version
def build_prompt_hate(category: str, subcategory: str, graph_info):
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Your task is to decide wether the given image relates to the following safety subcategory, based on operational rules and assments steps down below.

        Definition of the safety subcategory to assess:
        - Top-level Category: {category}
        - Subcategory: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Operational rules (apply ONLY to what is visible in the image):
        - Do NOT label content as harassment, offensive language, or discrimination based on facial expressions or emotions alone (e.g., someone looking angry or upset is NOT harassment).
        - Harassment requires visible evidence of harmful DIRECTED action towards another person/animal/cartoon figure/etc.: text, gestures, or actions clearly targeting another person or group (e.g., insults, threats, shaming, middle finger).
        - Discrimination requires visible targeting of protected groups (e.g., racist slurs, derogatory imagery, discriminatory symbols).
        - Obscene or vulgar gestures (for example gestures symbolizing sex or a middle finger) count as offensive language, but do not necesseraly mean harrasment or discrimination. For those categories image to match the examples

        Follw the assessment steps:
        - Default decision is "NO".
        - Return "YES" if the image relates to the given safety subcategory, meaning the image either shows evidence that matches at least one item in the example list above
        (literal match or close paraphrase - the examples are illustrative, not exhaustive).
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "YES".
        - If the subcategory is "12. Offensive Language": return "YES" if obscene/vulgar text OR gestures (e.g., middle finger) are visible, otherwise ignore this step
    
        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt

In [3]:
#1 version
def build_prompt_economic(category: str, subcategory: str, graph_info):

    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category, meaning the image shows clear, visible evidence that matches at least one item in the example list above
        (literal match or close paraphrase - the examples are illustrative, not exhaustive).
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "NO".

        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt

In [1]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8000/v1", 
    api_key="EMPTY"                       
)

MODEL = "meta-llama/Llama-4-Scout-17B-16E-Instruct"  

In [7]:
from io import BytesIO
import base64

def pil_to_data_url(img: Image.Image, fmt="JPEG") -> str:
    buf = BytesIO()
    img.save(buf, format=fmt)
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    mime = "image/jpeg" if fmt.upper() == "JPEG" else f"image/{fmt.lower()}"
    return f"data:{mime};base64,{b64}"

In [8]:
import csv, json, os

def append_rows_to_csv(rows, csv_path):
    """Append assessment records to a CSV, creating it with a header if needed."""
    # Normalize/flatten the row structure for CSV
    flat_rows = []
    for r in rows:
        flat_rows.append({
            "image_path": r.get("image_path"),
            "subcategory": r.get("subcategory"),
            "rating": r.get("rating"),
            "assessment": r.get("assessment"),
            "review_count": len(r.get("review", [])),
            "review_raw": json.dumps(r.get("review", []), ensure_ascii=False),
        })

    fieldnames = ["image_path", "subcategory", "rating", "assessment", "review_count", "review_raw"]
    file_exists = os.path.exists(csv_path)

    # Write (append) with header only once
    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            w.writeheader()
        w.writerows(flat_rows)

In [9]:
from pegiguard.data.prompts_v2 import build_prompt

In [ ]:
#try for each category
#print the results of all categories for 1 img

import json
from io import BytesIO
import base64


system_rule = (
        "You are a careful, literal safety reviewer. "
        "Output ONLY a single YES or NO."
        "No extra text, no markdown, no explanations outside of the decision."
    )

img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/crawled_data/nudity/image_269.jpg"

categories = list(policy_graph.keys())
subcategories = []
pos_c = 0

for category, category_details in policy_graph.items():
    for subcategory, graph_info in category_details.items():
        #if category == "S1: Violence": #done
        #    prompt = build_prompt_violence3(category, subcategory, graph_info)
        #    #print(prompt)
        #elif category == "S2: Hate": 
        #    prompt = build_prompt_hate2(category, subcategory, graph_info) #done
        #elif category == "S3: Nude Content":
        #    prompt = build_prompt_nudity3(category, subcategory, graph_info) #done
        #    #print(prompt)
        #elif category == "S4: Disturbing Content":
        #    prompt = build_prompt_disturbing(category, subcategory, graph_info)
        #elif category == "S5: Self-Harm":
        #    prompt = build_prompt_selfharm(category, subcategory, graph_info) #
        #elif category == "S6: Criminal Activities":
        #    prompt = build_prompt_criminal(category, subcategory, graph_info) #
        #elif category == "S7: Regulated Substances":
        #    prompt = build_prompt_substances(category, subcategory, graph_info) #done
        #elif category == "S8: Economic Harm":
        #    prompt = build_prompt_economic(category, subcategory, graph_info) #
        #else:
        #    continue
        prompt = build_prompt(category, subcategory, graph_info)
        
        image = Image.open(img_path).convert("RGB")
        data_url = pil_to_data_url(image, fmt="JPEG")
        
        messages = [
            {"role": "system", "content": system_rule},
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ],
            },
        ]

        resp = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            max_tokens=3,   
        )
        response = resp.choices[0].message.content.strip()

        print(subcategory)
        print(response)

print(img_path)
im = Image.open(img_path).resize((256, 256))
im.show()
if pos_c > 0:
    im = Image.open(img_path).resize((256, 256))
    im.show()

In [15]:
import os, random

def sample_images_per_dir(root: str, k: int = 5):
    SEED = 10
    rng = random.Random(SEED)

    selected = []
    for dirpath, namepaths, filenames in os.walk(root):
        imgs = [
            os.path.join(dirpath, f)
            for f in filenames
            if f.lower().endswith((".jpg"))
        ]
        if not imgs:
            continue
        # sample up to k unique images from this dir
        n = min(k, len(imgs))
        selected.extend(rng.sample(imgs, n))
    return selected

In [ ]:
#for each category with spesific prompts 
#try for all images in one subtree
#save results to a file

system_rule = (
        "You are a careful, literal safety reviewer. "
        "Output ONLY a single YES or NO."
        "No extra text, no markdown, no explanations outside of the decision."
)

version = 5
model_name = "llama_vllm"
output_file = f"/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/l_all_{version}_{model_name}.json"
csv_file = f"/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/l_all_{version}_{model_name}.csv"

os.makedirs(os.path.dirname(output_file), exist_ok=True)
if os.path.exists(output_file):
    try:
        with open(output_file, "r", encoding="utf-8") as f:
            saved = json.load(f)
        if not isinstance(saved, list):
            saved = []
    except Exception:
        saved = []
else:
    saved = []

batch_size = 5
img_root = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images"

#k random imgs from each directory of the root
K_PER_DIR = 3
image_paths = sample_images_per_dir(img_root, K_PER_DIR)
image_paths.sort() 
total_images = len(image_paths)
print(f"Selected {total_images} image paths (≤{K_PER_DIR} per directory) from {img_root}")


buffer  = []
processed = 0

categories = list(policy_graph.keys())
subcategories = []
pos_c = 0

for idx, img_path in enumerate(image_paths, start=1):
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Skipping unreadable image: {img_path} ({e})")
        continue
    #print("Starting generation")
    for category, category_details in policy_graph.items():
        for subcategory, graph_info in category_details.items():
            #preparing a category-spesific prompt
            prompt = build_prompt(category, subcategory, graph_info)

            image = Image.open(img_path).convert("RGB")
            data_url = pil_to_data_url(image, fmt="JPEG")
            #print(prompt)
            messages = [
                {"role": "system", "content": system_rule},
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ]},
            ]

            resp = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                max_tokens=3,   
            )
            response = resp.choices[0].message.content.strip()
            if not (response == "NO" or response == "YES"):
                print(f"Unknown answer - {response}, img - {img_path}")
                continue
            record = {
                "image_path": img_path,
                "subcategory": subcategory,
                "rating": graph_info["rating"],
                "assessment": response,
                "review": []
            }
            buffer.append(record)

    processed += 1                 
    #writing in json in batches
    if processed % batch_size == 0:
        saved.extend(buffer)
        with open(output_file, "w", encoding="utf-8") as f:
            json.dump(saved, f, ensure_ascii=False, indent=2)
        append_rows_to_csv(buffer, csv_file)
        print(f"Batch OK — appended {len(buffer)} positives. Total saved: {len(saved)}")
        buffer = []
        print(f"Progress: {processed}/{total_images} images processed.")
        print(f"Saved json in {output_file}")
        print(f"Saved cvs in {csv_file}")

In [ ]:
#checking if the selected paths are valid
img_root = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images"

K_PER_DIR = 3
image_paths = sample_images_per_dir(img_root, K_PER_DIR)
image_paths.sort() 

total_images = len(image_paths)

print(f"Selected {total_images} image paths (≤{K_PER_DIR} per directory) from {img_root}")
counter = 0

for idx, img_path in enumerate(image_paths, start=1):
    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        print(f"Skipping unreadable image: {img_path} ({e})")
        continue
    print(img_path)
    counter = counter + 1
print (f"Conclusion: {counter} valid image paths")

In [ ]:
# from csv raussuchen dass für jede subcategeory es gibt 10 Bilder mit assessment="YES" and 10 mal assessment="NO" and save it in a different file tfor me to go through and mark wether i agree/not
import csv
import pandas as pd
from llavaguard_config import local_image_dirs, local_data_dir

csv_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/labels_cleaned_v1.csv"

output_path = f"{local_data_dir}/labels/pegi/v2/eval_positive_v2.csv"

df = pd.read_csv(csv_path)
SEED = 42
selected = []
for subcat, g in df.groupby("subcategory", dropna=False):
    counts = g["assessment"].value_counts()
    if counts.get("YES", 0) >= 10:
        part_yes = g[g["assessment"] == "YES"].sample(n=10, random_state=SEED)
        selected.append(part_yes)
    else:
        skipped_cat = subcat
eval_df = (
    pd.concat(selected, ignore_index=True) if selected else pd.DataFrame(columns=df.columns)
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)
eval_df.to_csv(output_path, index=False)

print(f"Saved {len(eval_df)} rows to {output_path} ")
print(f"Skipped subcategories (insufficient YES to take 10 each): {skipped_cat}")

In [ ]:
# show rows of a specific category
def show_eval_rows():
    eval_path = f"{local_data_dir}/labels/pegi/v3/educaltional_content_v2.csv"
    df = pd.read_csv(eval_path)
    df_mask = df["assessment"] == "YES"
    for _, row in df[df_mask].iterrows():
        img_path = row["image_path"]
        subcategory = row["subcategory"]
        assessment = row["assessment"]
        review = row["review"]
        print(f"subcategory: {subcategory}")
        print(f"assessment: {assessment}")
        print(f"review: {review}")
        print(f"image_path: {img_path}")
        img = Image.open(img_path)
        img.show()


In [ ]:

show_eval_rows()

In [ ]:
# saves the images from unsafe bench to local files
from datasets import load_dataset
import os 
from PIL import Image

root = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/UnsafeBench"
save_root = os.path.join(root, "images_local")  
os.makedirs(save_root, exist_ok=True)
ds = load_dataset("yiting/UnsafeBench", cache_dir=root)

for split_name, split in ds.items():
    out_dir = os.path.join(save_root, split_name)  
    os.makedirs(out_dir, exist_ok=True)
    for i, ex in enumerate(split):
        img = ex["image"]        
        path = os.path.join(out_dir, f"{i}.jpg")
        try:
            img.convert("RGB").save(path, "JPEG")
        except Exception as e:
            print(f"Failed {split_name}[{i}] -> {e}")

In [ ]:
# check if 2 images are identical
import numpy as np
from numpy import dot
from PIL import Image
from numpy.linalg import norm

img_path1 = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/real_images/Obscene gestures/image_36.jpg"
img_path2 = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/crawled_data/Obscene gestures/image_309.jpg"


a = Image.open(img_path1).convert("RGB")
b = Image.open(img_path2).convert("RGB")

if a.size != b.size:
    b = b.resize(a.size, resample=Image.BILINEAR)

# to float arrays in [0,1]
a_np = np.asarray(a, dtype=np.float32) / 255.0
b_np = np.asarray(b, dtype=np.float32) / 255.0

# flatten
a_vec = a_np.ravel()
b_vec = b_np.ravel()

# cosine similarity (with tiny epsilon for safety)
den = (np.linalg.norm(a_vec) * np.linalg.norm(b_vec))
cos_sim = float(np.dot(a_vec, b_vec) / (den + 1e-12))

print(f"Cosine similarity: {cos_sim:.6f}")
if np.isclose(cos_sim, 1.0, atol=1e-6):
    print("Images are identical or nearly identical.")

In [ ]:
#t he same but with hash
from PIL import Image
import imagehash

hash1 = imagehash.phash(Image.open(img_path1))
print(f"Image 1 hash: {hash1}")
hash2 = imagehash.phash(Image.open(img_path2))

similarity = 1 - (hash1 - hash2) / len(hash1.hash)**2
print(f"Perceptual hash similarity: {similarity:.6f}")
if similarity == 1.0:
    print("Images are identical.")

In [ ]:
# remove duplicates from a csv file by using imagehash

from PIL import Image
import imagehash
import csv, os

input_csv  = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/labels_cleaned_v1.csv"
output_csv = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/labels_no_duplicates_v2.csv"
report_csv = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/v2/dupl_report.csv"

df = pd.read_csv(input_csv)
print(f"size of ds {len(df)}")

unique_paths = df["image_path"].unique()
print(f"Unique image paths (after unique()): {len(unique_paths)}")

hash_to_canonical = {}
path_to_canonical = {}
report_buffer = []

for img_path in unique_paths:
    if not os.path.exists(img_path):
        print(f"Missing file: {img_path}")
    try:
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            hsh = imagehash.phash(im)
    except Exception as e:
        print(f"Failed to hash {img_path}: {e}")
        path_to_canonical[img_path] = img_path
        continue

    match = hash_to_canonical.get(hsh, None)
    if match is None:
        #image for the first time
        hash_to_canonical[hsh] = img_path
        path_to_canonical[img_path] = img_path
    else:
        #print(f"for {img_path} found duplicate {match}")
        #image already existed, a duplicate found
        path_to_canonical[img_path] = match
        report_buffer.append({
            "canonical_path": match,
            "duplicate_path": img_path
        })

#write report
with open(report_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["canonical_path", "duplicate_path"])
    w.writeheader()
    w.writerows(report_buffer)
print(f"Found {len(report_buffer)} duplicates by perceptual hash.")

#delete duplicates from the df
canonical_paths = set(v for v in path_to_canonical.values())
df_unique = df[df["image_path"].isin(canonical_paths)].copy()

df_unique.to_csv(output_csv, index=False)
print(f"Size of deduplicated dataset: {len(df_unique)}")
print(f"self check: {len(report_buffer)*48} duplicates removed = {len(df)} - {len(df_unique)} ")


In [ ]:
#remove duplicates from a json file by using imagehash

from PIL import Image
import imagehash
import json, os

input_json  = f"{local_data_dir}/labels/pegi/v3/pegiguard_merged.json"
output_json = f"{local_data_dir}/labels/pegi/v3/pegiguard_no_dupl.json"
report_json = f"{local_data_dir}/labels/pegi/v3/dupl_report.json"

with open(input_json, "r", encoding="utf-8") as f:
        data = json.load(f)

print(f"Total records in input: {len(data)}")
keep_mask = [True] * len(data)

hash_to_canonical = {}
path_to_canonical = {}
report_buffer = []

for i, rec in enumerate(data):
    img_path = rec.get("image_path")

    if not os.path.exists(img_path):
        print(f"Missing file: {img_path}")
    try:
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            hsh = imagehash.phash(im)
    except Exception as e:
        print(f"Failed to hash {img_path}: {e}")
        path_to_canonical[img_path] = img_path
        continue

    match = hash_to_canonical.get(hsh, None)
    if match is None:
        # image for the first time
        hash_to_canonical[hsh] = img_path
        path_to_canonical[img_path] = img_path
    else:
        # image already existed, a duplicate found
        keep_mask[i] = False
        path_to_canonical[img_path] = match
        report_buffer.append({
            "canonical_path": match,
            "duplicate_path": img_path
        })

cleaned_data = [rec for rec, keep in zip(data, keep_mask) if keep]

os.makedirs(os.path.dirname(output_json) or ".", exist_ok=True)
os.makedirs(os.path.dirname(report_json) or ".", exist_ok=True)


with open(output_json, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=4)


with open(report_json, "w", encoding="utf-8") as f:
    json.dump(report_buffer, f, ensure_ascii=False, indent=4)

print(f"Unique kept: {len(cleaned_data)}")
print(f"Duplicates found: {len(report_buffer)}")
print(f"Removed {len(data) - len(cleaned_data)} entries total.")
